In [1]:
# Load env variables and create NVIDIA client
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("NVIDIA_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://integrate.api.nvidia.com/v1",
)

model = "qwen/qwen3-next-80b-a3b-instruct"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages.extend(messages)

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": all_messages,
        "temperature": temperature,
    }

    if stop_sequences:
        params["stop"] = stop_sequences

    resp = client.chat.completions.create(**params)
    return resp.choices[0].message.content

In [6]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 6 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [7]:
# Generate the dataset and write it to 'dataset_nvidia.json'
dataset = generate_dataset()
with open("dataset_nvidia.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(json.dumps(dataset, indent=2))

[
  {
    "task": "Extract all AWS account IDs from a log string using a regex. AWS account IDs are 12-digit numbers.",
    "format": "regex",
    "solution_criteria": "Regex must match exactly 12 consecutive digits and nothing else; must not match partial IDs or numbers with more/less than 12 digits."
  },
  {
    "task": "Generate a JSON object that defines an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'.",
    "format": "json",
    "solution_criteria": "JSON must be a valid IAM policy document with Version '2012-10-17', Effect 'Allow', Action 's3:Get*', and Resource arn:aws:s3:::my-data-bucket/* and arn:aws:s3:::my-data-bucket."
  },
  {
    "task": "Write a Python function that takes an AWS region name and returns True if it's a valid AWS commercial region (e.g., 'us-east-1', 'eu-west-2'), False otherwise.",
    "format": "python",
    "solution_criteria": "Function must return True for standard AWS regions like 'us-east-1', 'ap-southeast-

In [8]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [9]:
# Passes a test case into the model
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [10]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    fmt = test_case["format"]
    if fmt == "json":
        return validate_json(response)
    elif fmt == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [11]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [12]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [13]:
with open("dataset_nvidia.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.5


In [14]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\\b\\d{12}\\b",
    "test_case": {
      "task": "Extract all AWS account IDs from a log string using a regex. AWS account IDs are 12-digit numbers.",
      "format": "regex",
      "solution_criteria": "Regex must match exactly 12 consecutive digits and nothing else; must not match partial IDs or numbers with more/less than 12 digits."
    },
    "score": 9.0,
    "reasoning": "The regex correctly enforces 12 consecutive digits and avoids matching numbers of different lengths, but \\b relies on word/non-word character transitions, which may fail in log contexts with punctuation or formatting. A more robust solution would anchor the match with lookarounds or explicit delimiters."
  },
  {
    "output": "{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:ListBucket\"\n      ],\n      \"Resource\": [\n        \"arn:aws:s3:::my-data-bucket\",\n        \"arn:aws:s3:::m